In [ ]:
"""
Smart Device Management System
Course: EL 162 / 234 Object Oriented Programming
Author: Undergraduate Student
"""

class SmartDevice:
    """Parent class representing a baseline smart home device."""
    def __init__(self, device_id: str, name: str):
        if not device_id or not device_id.strip():
            raise ValueError("Device ID cannot be empty or blank.")

        # Encapsulated private attributes
        self.__device_id = device_id.strip()
        self.__power_status = False  # Devices start powered off by default

        # Public attribute
        self.name = name

    # --- Getters and Setters ---
    @property
    def device_id(self) -> str:
        return self.__device_id

    @property
    def power_status(self) -> bool:
        return self.__power_status

    @power_status.setter
    def power_status(self, status: bool):
        if isinstance(status, bool):
            self.__power_status = status
        else:
            print("Error: Power status must be a boolean value (True/False).")

    # --- Core Methods ---
    def turn_on(self):
        """Switches the device state to powered on."""
        self.__power_status = True
        print(f"[{self.name}] has been powered ON.")

    def turn_off(self):
        """Switches the device state to powered off."""
        self.__power_status = False
        print(f"[{self.name}] has been powered OFF.")

    def display_info(self):
        """Prints diagnostic details of the base device."""
        status_str = "ON" if self.__power_status else "OFF"
        print(f"Device Name: {self.name}")
        print(f"ID Ref:      {self.__device_id}")
        print(f"Power State: {status_str}")


class TemperatureSensor(SmartDevice):
    """Child class specialized for environmental monitoring."""
    def __init__(self, device_id: str, name: str, initial_temp: float = 22.5):
        # Forwarding initialization to the parent class via super()
        super().__init__(device_id, name)
        self.temperature = initial_temp

    def read_temperature(self) -> float:
        """Simulates reading ambient temperature updates."""
        if not self.power_status:
            print(f"Operation Failed: {self.name} is powered off.")
            return 0.0
        print(f"Reading from {self.name}: Current ambient temperature is {self.temperature}°C.")
        return self.temperature


class SmartLight(SmartDevice):
    """Child class controlling home illumination levels."""
    def __init__(self, device_id: str, name: str, default_brightness: int = 50):
        super().__init__(device_id, name)
        self._brightness = 0
        self.brightness = default_brightness  # Triggers the validation logic in setter

    @property
    def brightness(self) -> int:
        return self._brightness

    @brightness.setter
    def brightness(self, value: int):
        # Strict validation constraint: enforcing 0 to 100 boundaries
        if 0 <= value <= 100:
            self._brightness = value
        else:
            print(f"Invalid Entry: Brightness level ({value}) must remain within 0-100.")

    def increase_brightness(self, amount: int = 10):
        """Increments the brightness level by a specified scale factor."""
        if not self.power_status:
            print(f"Operation Failed: Cannot alter settings. {self.name} is offline.")
            return

        target = self._brightness + amount
        if target > 100:
            self.brightness = 100
            print(f"[{self.name}] Hit ceiling limit. Brightness locked at 100%.")
        else:
            self.brightness = target
            print(f"[{self.name}] Dimming increased to {self.brightness}%.")

    def decrease_brightness(self, amount: int = 10):
        """Decrements the brightness level by a specified scale factor."""
        if not self.power_status:
            print(f"Operation Failed: Cannot alter settings. {self.name} is offline.")
            return

        target = self._brightness - amount
        if target < 0:
            self.brightness = 0
            print(f"[{self.name}] Hit baseline floor. Brightness locked at 0%.")
        else:
            self.brightness = target
            print(f"[{self.name}] Dimming dropped to {self.brightness}%.")


class SecurityCamera(SmartDevice):
    """Child class managed for home perimeter recording loops."""
    def __init__(self, device_id: str, name: str):
        super().__init__(device_id, name)
        self.recording_status = False

    def start_recording(self):
        """Enables the device recording stream."""
        if not self.power_status:
            print(f"Operation Failed: {self.name} is offline. Turn power on first.")
            return
        self.recording_status = True
        print(f"[{self.name}] Video stream initialized. Recording live...")

    def stop_recording(self):
        """Disables the device recording stream."""
        self.recording_status = False
        print(f"[{self.name}] Video stream terminated. Recording stopped.")


def run_interactive_dashboard():
    """Builds and hosts the command-line interface execution layer."""
    # Instantiating required test hardware instances
    thermal_node = TemperatureSensor("TS-404", "Living Room Thermal Sensor", 24.2)
    lounge_bulb = SmartLight("SL-777", "Lounge Chandelier", 75)
    patio_cam = SecurityCamera("CAM-909", "Backyard Security Camera")

    # Grouping instances inside an indexable array structure
    inventory = [thermal_node, lounge_bulb, patio_cam]

    while True:
        print("\n" + "=" * 45)
        print("     SMART HOME DEVICE SYSTEM DASHBOARD     ")
        print("=" * 45)
        print("1. Display All Device Inventories")
        print("2. Power ON a Device")
        print("3. Power OFF a Device")
        print("4. Poll Temperature Data")
        print("5. Modify Light Brightness Scale")
        print("6. Manage Camera Feed Recording")
        print("7. Terminate Application Pipeline")
        print("-" * 45)

        choice = input("Enter option sequence (1-7): ").strip()

        if choice == "1":
            print("\n--- Current Systems Registry ---")
            for idx, item in enumerate(inventory, 1):
                print(f"\n[{idx}]", end=" ")
                item.display_info()
                if isinstance(item, TemperatureSensor):
                    print(f"Current Value: {item.temperature}°C")
                elif isinstance(item, SmartLight):
                    print(f"Dimmer Step:   {item.brightness}%")
                elif isinstance(item, SecurityCamera):
                    state_lbl = "Capturing Feed" if item.recording_status else "Standby Mode"
                    print(f"Capture State: {state_lbl}")

        elif choice in ["2", "3"]:
            print("\nSelect target device infrastructure:")
            for idx, item in enumerate(inventory, 1):
                print(f"{idx}. {item.name} ({item.device_id})")

            try:
                target_idx = int(input("Select target: ")) - 1
                if 0 <= target_idx < len(inventory):
                    if choice == "2":
                        inventory[target_idx].turn_on()
                    else:
                        # Make sure to shut down recording loops if cutting camera power
                        if isinstance(inventory[target_idx], SecurityCamera):
                            inventory[target_idx].stop_recording()
                        inventory[target_idx].turn_off()
                else:
                    print("Selection index out of bounds.")
            except ValueError:
                print("Invalid input. Numeric value expected.")

        elif choice == "4":
            thermal_node.read_temperature()

        elif choice == "5":
            if not lounge_bulb.power_status:
                print(f"Operation Failed: {lounge_bulb.name} must be turned ON first.")
                continue
            print(f"\nTargeting: {lounge_bulb.name} (Current: {lounge_bulb.brightness}%)")
            print("1. Increment Brightness (+10%)")
            print("2. Decrement Brightness (-10%)")
            sub_choice = input("Select operation (1-2): ").strip()
            if sub_choice == "1":
                lounge_bulb.increase_brightness()
            elif sub_choice == "2":
                lounge_bulb.decrease_brightness()
            else:
                print("Unknown command choice passed.")

        elif choice == "6":
            if not patio_cam.power_status:
                print(f"Operation Failed: {patio_cam.name} must be turned ON first.")
                continue
            print(f"\nTargeting: {patio_cam.name}")
            print("1. Begin Recording Thread")
            print("2. Terminate Recording Thread")
            sub_choice = input("Select operation (1-2): ").strip()
            if sub_choice == "1":
                patio_cam.start_recording()
            elif sub_choice == "2":
                patio_cam.stop_recording()
            else:
                print("Unknown command choice passed.")

        elif choice == "7":
            print("\nShutting down Management Dashboard. Goodbye.")
            break
        else:
            print("Option choice not recognized. Please provide an option index from 1 to 7.")

if __name__ == "__main__":
    run_interactive_dashboard()